# Week 7 — Advanced Models

## Goal

This notebook will:

- load the Week 6 feature data;
- train an **XGBoost** home-price model;
- try a small number of model settings;
- choose the best settings using a validation set;
- report the final test results;
- save the Week 7 result files in `results/07/`.

The main test measures are **R², MAE, RMSE, and MdAPE**.

## 1. Import the packages

The code below installs XGBoost only when it is missing.

In [ ]:
from pathlib import Path
import subprocess
import sys
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

try:
    from xgboost import XGBRegressor
except ImportError:
    print("XGBoost is not installed. Installing it now...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost"])
    from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

print("Packages are ready.")

## 2. Set the file paths

The notebook expects the Week 6 file at:

`results/06/feature_engineered_data.csv`

All new files will be saved in:

`results/07/`

In [ ]:
working_dir = Path.cwd()

# This works when the notebook is in the main project folder
# or inside a folder named "notebooks".
project_root = (
    working_dir.parent
    if working_dir.name.lower() == "notebooks"
    else working_dir
)

data_path = project_root / "results" / "06" / "feature_engineered_data.csv"
results_dir = project_root / "results" / "07"
results_dir.mkdir(parents=True, exist_ok=True)

print("Input file:", data_path)
print("Output folder:", results_dir)

## 3. Load the Week 6 data

In [ ]:
if not data_path.exists():
    raise FileNotFoundError(
        f"Cannot find {data_path}. "
        "Run the Week 6 notebook first and make sure "
        "feature_engineered_data.csv is inside results/06/."
    )

df = pd.read_csv(data_path, low_memory=False)
df.columns = df.columns.str.strip()

print("Data shape:", df.shape)
display(df.head())

## 4. Find the important columns

The code checks a few common column names. This makes the notebook work with small name differences such as `ClosePrice` versus `SalePrice`.

In [ ]:
def first_existing(candidates):
    # Return the first column name that exists in the data.
    for column in candidates:
        if column in df.columns:
            return column
    return None


def clean_number(series):
    # Change money or text values into numbers.
    return pd.to_numeric(
        series.astype(str).str.replace(r"[^0-9.\-]", "", regex=True),
        errors="coerce",
    )


target_col = first_existing(
    ["ClosePrice", "SoldPrice", "SalePrice", "CurrentPrice", "price"]
)

bed_col = first_existing(
    ["BedroomsTotal", "Bedrooms", "BedsTotal", "Beds", "bedrooms"]
)

bath_col = first_existing(
    [
        "BathroomsTotalInteger",
        "BathroomsTotalDecimal",
        "BathroomsFull",
        "Bathrooms",
        "BathsTotal",
        "Baths",
        "bathrooms",
    ]
)

year_built_col = first_existing(
    ["YearBuilt", "year_built", "BuiltYear"]
)

bed_bath_col = first_existing(
    ["bed_bath_ratio", "BedBathRatio"]
)

property_age_col = first_existing(
    ["property_age", "PropertyAge"]
)

school_district_col = first_existing(
    ["DistrictName", "SchoolDistrict", "SchoolDistrictName", "district_name"]
)

required_columns = {
    "target price": target_col,
    "bed/bath ratio": bed_bath_col,
    "property age": property_age_col,
    "school district": school_district_col,
}

missing_items = [name for name, column in required_columns.items() if column is None]

if missing_items:
    raise ValueError(
        "The Week 6 data is missing these required items: "
        f"{missing_items}. Please check the Week 6 output file."
    )

print("Target column:", target_col)
print("Bed/bath ratio column:", bed_bath_col)
print("Property age column:", property_age_col)
print("School district column:", school_district_col)

## 5. Choose the model features

The model uses the same type of property features used in Week 6. It also includes the three new Week 6 features:

- bed/bath ratio;
- property age;
- school district.

Only columns that are actually in the data are kept.

In [ ]:
numeric_candidates = [
    bed_col,
    bath_col,
    year_built_col,
    "LivingArea",
    "BuildingAreaTotal",
    "LotSizeSquareFeet",
    "LotSizeArea",
    "Latitude",
    "Longitude",
    "ListPrice",
    "OriginalListPrice",
    "DaysOnMarket",
    "CumulativeDaysOnMarket",
    "GarageSpaces",
    "Stories",
    bed_bath_col,
    property_age_col,
]

numeric_features = []

for column in numeric_candidates:
    if (
        column is not None
        and column in df.columns
        and column != target_col
        and column not in numeric_features
    ):
        numeric_features.append(column)
        df[column] = clean_number(df[column])

categorical_features = [school_district_col]

df[target_col] = clean_number(df[target_col])
df[school_district_col] = (
    df[school_district_col]
    .replace("", np.nan)
    .fillna("Unknown")
    .astype(str)
)

if not numeric_features:
    raise ValueError("No numeric model features were found.")

print("Numeric features:")
print(numeric_features)
print("\nCategory feature:")
print(categorical_features)

## 6. Prepare the model data

Rows without a valid positive sale price are removed.

In [ ]:
model_df = df[
    df[target_col].notna()
    & (df[target_col] > 0)
].copy()

feature_columns = numeric_features + categorical_features

X = model_df[feature_columns].copy()
y = model_df[target_col].copy()

print("Rows used for modeling:", len(model_df))
print("Number of raw features:", len(feature_columns))

## 7. Make the training and test sets

When the data already has a `split` column with `train` and `test`, the notebook keeps that split.

When there is no usable `split` column, it uses an 80% training and 20% test split with `random_state=42`.

In [ ]:
split_col = first_existing(["split", "Split"])
use_saved_split = False

if split_col is not None:
    split_values = model_df[split_col].astype(str).str.strip().str.lower()
    use_saved_split = {"train", "test"}.issubset(set(split_values.unique()))

if use_saved_split:
    train_mask = split_values == "train"
    test_mask = split_values == "test"

    X_train = X.loc[train_mask].copy()
    X_test = X.loc[test_mask].copy()
    y_train = y.loc[train_mask].copy()
    y_test = y.loc[test_mask].copy()

    split_method = "Saved train/test split"
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
    )

    split_method = "Random 80/20 split"

if X_train.empty or X_test.empty:
    raise ValueError("The training set or test set is empty.")

print("Split method:", split_method)
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

## 8. Create a small validation set

The validation set is used to choose the model settings. The final test set is not used during this step.

In [ ]:
X_fit, X_valid, y_fit, y_valid = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42,
)

print("Rows used to fit each trial model:", len(X_fit))
print("Validation rows:", len(X_valid))

## 9. Prepare numeric and category data

Missing numeric values are filled with the median. School district names are changed into 0/1 columns.

XGBoost does not need numeric features to be scaled.

In [ ]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

category_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("school_district", category_pipeline, categorical_features),
    ]
)

X_fit_ready = preprocessor.fit_transform(X_fit)
X_valid_ready = preprocessor.transform(X_valid)

print("Prepared training shape:", X_fit_ready.shape)
print("Prepared validation shape:", X_valid_ready.shape)

## 10. Try a few XGBoost settings

The three settings below control the model:

- `max_depth`: how deep each tree can grow;
- `learning_rate`: how large each update is;
- `n_estimators`: how many trees are used.

This is a light search with only six combinations.

In [ ]:
parameter_settings = [
    {"max_depth": 3, "learning_rate": 0.05, "n_estimators": 150},
    {"max_depth": 3, "learning_rate": 0.10, "n_estimators": 100},
    {"max_depth": 5, "learning_rate": 0.05, "n_estimators": 200},
    {"max_depth": 5, "learning_rate": 0.10, "n_estimators": 150},
    {"max_depth": 7, "learning_rate": 0.05, "n_estimators": 200},
    {"max_depth": 7, "learning_rate": 0.10, "n_estimators": 100},
]

parameter_settings

In [ ]:
tuning_results = []

for setting in parameter_settings:
    trial_model = XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
        tree_method="hist",
        subsample=0.90,
        colsample_bytree=0.90,
        verbosity=0,
        **setting,
    )

    trial_model.fit(X_fit_ready, y_fit)
    validation_prediction = trial_model.predict(X_valid_ready)

    validation_mae = mean_absolute_error(y_valid, validation_prediction)
    validation_rmse = np.sqrt(
        mean_squared_error(y_valid, validation_prediction)
    )
    validation_r2 = r2_score(y_valid, validation_prediction)

    tuning_results.append(
        {
            **setting,
            "Validation_MAE": validation_mae,
            "Validation_RMSE": validation_rmse,
            "Validation_R2": validation_r2,
        }
    )

    print(
        f"Finished: depth={setting['max_depth']}, "
        f"learning_rate={setting['learning_rate']}, "
        f"n_estimators={setting['n_estimators']}"
    )

print("All settings are finished.")

## 11. Select the best settings

The best setting has the lowest validation RMSE.

In [ ]:
tuning_results_df = (
    pd.DataFrame(tuning_results)
    .sort_values("Validation_RMSE")
    .reset_index(drop=True)
)

display(tuning_results_df)

tuning_results_path = results_dir / "xgboost_tuning_results.csv"
tuning_results_df.to_csv(tuning_results_path, index=False)

best_row = tuning_results_df.iloc[0]

best_params = {
    "max_depth": int(best_row["max_depth"]),
    "learning_rate": float(best_row["learning_rate"]),
    "n_estimators": int(best_row["n_estimators"]),
}

print("Best settings:")
print(best_params)
print("Saved:", tuning_results_path)

## 12. Train the final model

The final model uses all training rows and the best settings found above.

In [ ]:
final_preprocessor = clone(preprocessor)

X_train_ready = final_preprocessor.fit_transform(X_train)
X_test_ready = final_preprocessor.transform(X_test)

final_model = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
    subsample=0.90,
    colsample_bytree=0.90,
    verbosity=0,
    **best_params,
)

final_model.fit(X_train_ready, y_train)

train_prediction = final_model.predict(X_train_ready)
test_prediction = final_model.predict(X_test_ready)

print("Final XGBoost model is trained.")

## 13. Calculate the final test results

- **R²:** higher is better.
- **MAE:** average dollar error; lower is better.
- **RMSE:** gives more weight to large errors; lower is better.
- **MdAPE:** median percentage error; lower is better.

In [ ]:
def mdape(y_true, y_pred):
    # Return median absolute percentage error as a percent.
    true_values = np.asarray(y_true, dtype=float)
    predicted_values = np.asarray(y_pred, dtype=float)

    nonzero = true_values != 0

    return 100 * np.median(
        np.abs(
            (true_values[nonzero] - predicted_values[nonzero])
            / true_values[nonzero]
        )
    )


train_r2 = r2_score(y_train, train_prediction)
test_r2 = r2_score(y_test, test_prediction)
test_mae = mean_absolute_error(y_test, test_prediction)
test_rmse = np.sqrt(mean_squared_error(y_test, test_prediction))
test_mdape = mdape(y_test, test_prediction)

metrics_df = pd.DataFrame(
    [
        {
            "Model": "Tuned XGBoost",
            "Split_Method": split_method,
            "Training_Rows": len(X_train),
            "Test_Rows": len(X_test),
            "Raw_Feature_Count": len(feature_columns),
            "Best_Max_Depth": best_params["max_depth"],
            "Best_Learning_Rate": best_params["learning_rate"],
            "Best_N_Estimators": best_params["n_estimators"],
            "Train_R2": train_r2,
            "Test_R2": test_r2,
            "R2_Gap": train_r2 - test_r2,
            "Test_MAE": test_mae,
            "Test_RMSE": test_rmse,
            "Test_MdAPE_Percent": test_mdape,
        }
    ]
)

display(metrics_df)

metrics_path = results_dir / "xgboost_test_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)

print("Saved:", metrics_path)

## 14. Save the test predictions

In [ ]:
predictions_df = pd.DataFrame(
    {
        "Actual_Price": np.asarray(y_test),
        "Predicted_Price": test_prediction,
    },
    index=y_test.index,
)

predictions_df["Error"] = (
    predictions_df["Predicted_Price"]
    - predictions_df["Actual_Price"]
)

predictions_df["Absolute_Error"] = predictions_df["Error"].abs()

predictions_df["Absolute_Percentage_Error"] = (
    100
    * predictions_df["Absolute_Error"]
    / predictions_df["Actual_Price"].replace(0, np.nan)
)

predictions_df = predictions_df.reset_index(names="Original_Row_Index")

predictions_path = results_dir / "xgboost_test_predictions.csv"
predictions_df.to_csv(predictions_path, index=False)

display(predictions_df.head())
print("Saved:", predictions_path)

## 15. Plot actual price versus predicted price

Points closer to the dashed line are better predictions.

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, test_prediction, alpha=0.35)

line_min = min(float(np.min(y_test)), float(np.min(test_prediction)))
line_max = max(float(np.max(y_test)), float(np.max(test_prediction)))

plt.plot(
    [line_min, line_max],
    [line_min, line_max],
    linestyle="--",
)

plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Tuned XGBoost: Actual Price vs Predicted Price")
plt.tight_layout()

prediction_plot_path = results_dir / "xgboost_actual_vs_predicted.png"
plt.savefig(prediction_plot_path, dpi=150, bbox_inches="tight")
plt.show()

print("Saved:", prediction_plot_path)

## 16. Show the most important features

A larger importance value means the model used that feature more often to improve its predictions.

In [ ]:
feature_names = final_preprocessor.get_feature_names_out()

clean_feature_names = [
    name.replace("numeric__", "").replace("school_district__", "")
    for name in feature_names
]

importance_df = pd.DataFrame(
    {
        "Feature": clean_feature_names,
        "Importance": final_model.feature_importances_,
    }
).sort_values("Importance", ascending=False)

importance_path = results_dir / "xgboost_feature_importance.csv"
importance_df.to_csv(importance_path, index=False)

display(importance_df.head(20))
print("Saved:", importance_path)

In [ ]:
top_features = (
    importance_df.head(20)
    .sort_values("Importance", ascending=True)
)

plt.figure(figsize=(9, 7))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.title("Tuned XGBoost: Top 20 Features")
plt.tight_layout()

importance_plot_path = results_dir / "xgboost_feature_importance.png"
plt.savefig(importance_plot_path, dpi=150, bbox_inches="tight")
plt.show()

print("Saved:", importance_plot_path)

## 17. Save the final model

The saved file contains both the data preparation steps and the trained XGBoost model.

In [ ]:
model_bundle = {
    "preprocessor": final_preprocessor,
    "model": final_model,
    "target_column": target_col,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "best_parameters": best_params,
}

model_path = results_dir / "best_xgboost_model.joblib"
joblib.dump(model_bundle, model_path)

print("Saved:", model_path)

## 18. Final summary

After all cells run, `results/07/` will contain:

- `xgboost_tuning_results.csv`
- `xgboost_test_metrics.csv`
- `xgboost_test_predictions.csv`
- `xgboost_actual_vs_predicted.png`
- `xgboost_feature_importance.csv`
- `xgboost_feature_importance.png`
- `best_xgboost_model.joblib`

The main Week 7 answer is the table in `xgboost_test_metrics.csv`.

In [ ]:
print("Week 7 is complete.")
print("Best XGBoost settings:", best_params)
print(f"Test R²: {test_r2:.4f}")
print(f"Test MAE: ${test_mae:,.2f}")
print(f"Test RMSE: ${test_rmse:,.2f}")
print(f"Test MdAPE: {test_mdape:.2f}%")
print("All files were saved in:", results_dir)